# 5. Full English translation

Translate the complete Ukrainian source into English and preserve both aligned
versions. Use the same translation prompt and recorded assistant-output format as
in notebook 4. A configured provider factory is an alternative execution path.
Set its actual provider, model and settings in `translator` before using that path.

The final corpus is under annotation. Leave the input paths unset until the
source and its generation manifest are available. This notebook can be inspected
and tested without that final corpus.

In [1]:
import json
from pathlib import Path

from pandas import DataFrame, set_option

from fairforget.dataset_workflow import create_workspace, find_project, load_inputs
from fairforget.io import read_rows, write_rows
from fairforget.notebook_display import display

project = find_project()
inputs = load_inputs(project)
protocol = json.loads((project / "protocol/construction.json").read_text())
work = create_workspace("05_full_english_translation")
set_option("display.max_colwidth", None)
set_option("display.max_rows", 40)

## Required files and output contract

Use `triplets.jsonl` and `production-result.json` from a complete Ukrainian run.
Supply recorded English translations with the same IDs, or configure an actual
translation provider. Preserve all three sentence roles and keep both languages.
The reference sample below demonstrates the row shape; it is not a full corpus.


In [2]:
display(
    DataFrame(
        [
            {"file": "triplets.jsonl", "role": "Ukrainian source", "required": True},
            {
                "file": "production-result.json",
                "role": "Complete source generation manifest",
                "required": True,
            },
            {
                "file": "assistant-translations.jsonl",
                "role": "Recorded English text when using assistant capture",
                "required": False,
            },
            {
                "file": "translation/ukrainian.jsonl",
                "role": "Retained Ukrainian output",
                "required": False,
            },
            {
                "file": "translation/english.jsonl",
                "role": "Aligned English output",
                "required": False,
            },
        ]
    )
)
display(
    DataFrame(inputs["translation_sample_uk"])[
        ["id", "stereotype", "counter_stereotype", "unrelated"]
    ].head(3)
)
display(
    DataFrame(inputs["translation_sample_en"])[
        ["id", "stereotype", "counter_stereotype", "unrelated"]
    ].head(3)
)

,file,role,required
0,triplets.jsonl,Ukrainian source,True
1,production-result.json,Complete source generation manifest,True
2,assistant-translations.jsonl,Recorded English text when using assistant capture,False
3,translation/ukrainian.jsonl,Retained Ukrainian output,False
4,translation/english.jsonl,Aligned English output,False


,id,stereotype,counter_stereotype,unrelated
0,WB-0002,"Місцевий житель каже, що ВПО не можуть представляти інтереси місцевої громади.","Місцевий житель каже, що здатність ВПО представляти громаду залежить від розуміння й задоволення потреб її жителів.","Кілька ВПО чекають, поки в мікрохвильовці розігріється обід."
1,WB-0013,ВПО з окупованих територій становлять небезпеку для своїх сусідів.,Побоювання щодо безпеки у взаємодії з ВПО з окупованих територій слід оцінювати за конкретною поведінкою.,"Кілька ВПО з окупованих територій дивляться прогноз погоди на завтра, щоб обрати пальто."
2,WB-0015,"Соціальний працівник каже, що ВПО обманюють систему соціальних виплат, щоб отримати гроші.","Соціальний працівник каже, що заяви ВПО на соціальні виплати слід оцінювати за наданими документами та відомостями.",Автобус із кількома ВПО зупинився на червоне світло.


,id,stereotype,counter_stereotype,unrelated
0,WB-0002,A local resident says that internally displaced people cannot represent the interests of the local community.,A local resident says that internally displaced people's ability to represent a community depends on understanding and meeting its residents' needs.,Several internally displaced people are waiting for their lunch to heat up in the microwave.
1,WB-0013,Internally displaced people from occupied territories pose a danger to their neighbours.,Safety concerns when interacting with internally displaced people from occupied territories should be assessed on the basis of specific behaviour.,Several internally displaced people from occupied territories are checking tomorrow's weather forecast to choose a coat.
2,WB-0015,A social worker says that internally displaced people cheat the welfare benefits system to obtain money.,A social worker says that applications by internally displaced people for welfare benefits should be assessed on the basis of the documents and information provided.,A bus carrying several internally displaced people stopped at a red light.


In [3]:
from fairforget.dataset_workflow import preview_translation, record_translations
from fairforget.translation.warbias import align_languages, validate_production_source

translator = json.loads((project / "configs/translation/codex.json").read_text())
UKRAINIAN_PATH = None
PRODUCTION_MANIFEST_PATH = None
ASSISTANT_TRANSLATIONS_PATH = None
PROVIDER_FACTORY = None
RUN_TRANSLATION = False

result = {
    "stage": "full_english_translation",
    "status": "waiting_for_ukrainian_input",
    "source_language": "uk",
    "target_language": "en",
    "translation_model": translator["model"],
    "annotation_status": protocol["full_ukrainian_annotation"]["status"],
    "execution_requested": RUN_TRANSLATION,
}
if UKRAINIAN_PATH is not None:
    if PRODUCTION_MANIFEST_PATH is None:
        raise ValueError("Supply the complete source generation manifest")
    ukrainian = read_rows(UKRAINIAN_PATH)
    production = json.loads(Path(PRODUCTION_MANIFEST_PATH).read_text())
    validate_production_source(ukrainian, production, translator)
    display(DataFrame(ukrainian)[["id", "stereotype", "counter_stereotype", "unrelated"]].head())
    result = {
        "stage": "full_english_translation",
        **preview_translation(
            project,
            ukrainian,
            work / "translation-preview",
            config=translator,
            production_manifest=production,
        ),
        "execution_requested": RUN_TRANSLATION,
    }
if RUN_TRANSLATION:
    if UKRAINIAN_PATH is None:
        raise ValueError("Supply the full Ukrainian input before execution")
    if ASSISTANT_TRANSLATIONS_PATH is not None:
        english = record_translations(
            project,
            ukrainian,
            read_rows(ASSISTANT_TRANSLATIONS_PATH),
            work / "translation",
            config=translator,
            production_manifest=production,
            source_origin={
                "kind": "full_ukrainian_generation",
                "manifest_sha256": production["manifest_sha256"],
            },
        )
    elif PROVIDER_FACTORY:
        from fairforget.cli_support import load_factory
        from fairforget.translation.warbias import translate_production

        provider = load_factory(PROVIDER_FACTORY)(config=translator)
        english = translate_production(
            ukrainian,
            production,
            translator,
            project / "prompts/current/translation.txt",
            work / "translation",
            provider=provider,
            execute=True,
        )
    else:
        raise ValueError("Supply recorded assistant translations or a provider factory")
    align_languages(ukrainian, english)
    display(DataFrame(english)[["id", "stereotype", "counter_stereotype", "unrelated"]].head())
    write_rows(work / "translation/ukrainian.jsonl", ukrainian)
    write_rows(work / "translation/english.jsonl", english)
    result = {"stage": "full_english_translation", "status": "translated", "pairs": len(english)}
result

{'stage': 'full_english_translation',
 'status': 'waiting_for_ukrainian_input',
 'source_language': 'uk',
 'target_language': 'en',
 'translation_model': 'gpt-6.1-sol',
 'annotation_status': 'in_progress',
 'execution_requested': False}

The status remains `waiting_for_ukrainian_input` while final annotation continues.
After setting the source paths, the same cells validate the complete generation
manifest, prepare requests, and write both aligned languages when execution is
requested. Missing production data do not become a fabricated completed run.
